# Auto Lending Underwriting Fraud — Quickstart

Runs the full prototype in Google Colab: synthetic data → rules + ML fraud model → champion/challenger strategy → vendor scorecards → loss forecast → monthly CFO report and dashboard.

*All data is synthetic. Run the cells top to bottom (Runtime → Run all, about 2 minutes).*

In [ ]:
!git clone -q https://github.com/atlasse71/auto-fraud-risk-strategy
%cd auto-fraud-risk-strategy
!pip install -q -r requirements.txt

## 1. Run the full pipeline

In [ ]:
!python run_pipeline.py

## 2. Headline results: champion (legacy rules) vs challenger (rules + ML)

In [ ]:
import pandas as pd
pd.set_option('display.float_format', '{:,.3f}'.format)
cvc = pd.read_csv('outputs/champion_vs_challenger.csv').set_index('strategy')
cvc[['auto_approval_rate','manual_review_rate','fraud_capture_rate_usd','false_positive_rate','fraud_loss_bps','total_cost_usd']].T

## 3. Capture by fraud typology

In [ ]:
pd.read_csv('outputs/typology_capture.csv').pivot(index='typology', columns='strategy', values='capture_rate')

## 4. Vendor scorecard and emerging-vendor POC

In [ ]:
import json
vs = pd.read_csv('outputs/vendor_scorecard.csv')
display(vs[['vendor','standalone_auc','incremental_capture_pts','net_benefit_annual','roi','recommendation']])
poc = json.load(open('outputs/results.json'))['poc']
print(f"Vendor E POC: AUC {poc['auc_without']:.3f} -> {poc['auc_with']:.3f}, ROI {poc['roi']:.1f}x -> {poc['recommendation']}")

## 5. Try it yourself: change a cutoff and re-run

Edit the ML review cutoff in the policy file, re-run, and compare. Lower cutoffs catch more fraud but grow the review queue and add customer friction.

In [ ]:
import yaml
p = yaml.safe_load(open('config/decision_rules.yaml'))
p['model_cutoffs']['review_at_or_above'] = 0.30   # try 0.05, 0.15, 0.30
yaml.safe_dump(p, open('config/decision_rules.yaml', 'w'), sort_keys=False)
!python run_pipeline.py > /dev/null 2>&1
pd.read_csv('outputs/champion_vs_challenger.csv').set_index('strategy')[['manual_review_rate','fraud_capture_rate_usd','total_cost_usd']].T

## 6. Read the monthly report and open the dashboard

In [ ]:
from IPython.display import Markdown, display
import glob
display(Markdown(open(sorted(glob.glob('reports/*.md'))[-1]).read()))

In [ ]:
from google.colab import files
files.download('docs/index.html')   # opens offline in any browser

## 7. Run the tests

In [ ]:
!python -m pytest -q